# Módulo 6 · Clase 7 — Proyecto: Análisis Exploratorio de Datos (EDA)

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

```
Un EDA completo tiene 5 fases:

  1. CARGAR  → importar datos, entender estructura
  2. EXPLORAR → dimensiones, tipos, estadísticas iniciales
  3. LIMPIAR  → missing values, outliers, inconsistencias
  4. ANALIZAR → distribuciones, correlaciones, patrones
  5. CONCLUIR → preguntas de negocio respondidas, hipótesis para modelado

Dataset: Titanic (pasajeros del crucero)
Pregunta de negocio: ¿Qué factores determinan la supervivencia?
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 110)
print("Librerías listas")

---
## Fase 1 — Cargar datos

Usamos el dataset Titanic, disponible vía seaborn sin instalación extra.

In [ ]:
import seaborn as sns

# Dataset clásico: 891 pasajeros del Titanic
df_raw = sns.load_dataset('titanic')

print("=== Dataset cargado ===")
print(f"Dimensiones: {df_raw.shape}")
print(f"\nColumnas:")
for col in df_raw.columns:
    print(f"  {col:20} {df_raw[col].dtype}")

print("\nPrimeras filas:")
print(df_raw.head())

print("\nDescripción de las columnas:")
descripcion = {
    'survived':  'Sobrevivió (1=Sí, 0=No)',
    'pclass':    'Clase del billete (1=Primera, 2=Segunda, 3=Tercera)',
    'sex':       'Sexo del pasajero',
    'age':       'Edad en años',
    'sibsp':     'Nº de hermanos/cónyuge a bordo',
    'parch':     'Nº de padres/hijos a bordo',
    'fare':      'Tarifa del billete (£)',
    'embarked':  'Puerto de embarque (C=Cherburgo, Q=Queenstown, S=Southampton)',
    'class':     'Clase (texto)',
    'who':       'Categoría (man/woman/child)',
    'adult_male':'Hombre adulto (booleano)',
    'deck':      'Cubierta del camarote',
    'embark_town':'Ciudad de embarque',
    'alive':     'Sobrevivió (yes/no)',
    'alone':     'Viajaba solo',
}
for col, desc in descripcion.items():
    print(f"  {col:15}: {desc}")

---
## Fase 2 — Exploración inicial

In [ ]:
df = df_raw.copy()

print("=== Estadísticas descriptivas ===")
print(df.describe(include='all').round(2))

print("\n=== Missing values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct_%': missing_pct})
print(missing_df[missing_df['count'] > 0].sort_values('count', ascending=False))

print("\n=== Variable objetivo ===")
print(df['survived'].value_counts())
print(f"Tasa de supervivencia: {df['survived'].mean()*100:.1f}%")

print("\n=== Distribución de clases ===")
print(df['pclass'].value_counts().sort_index())

print("\n=== Distribución por sexo ===")
print(df['sex'].value_counts())

In [ ]:
# Panel de exploración inicial
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig)

# Supervivencia
ax0 = fig.add_subplot(gs[0, 0])
conteos = df['survived'].value_counts()
ax0.bar(['No sobrevivió', 'Sobrevivió'], conteos.values, color=['#e74c3c', '#2ecc71'])
ax0.set_title('Supervivencia')
for i, v in enumerate(conteos.values):
    ax0.text(i, v + 5, str(v), ha='center')

# Por clase
ax1 = fig.add_subplot(gs[0, 1])
df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Supervivencia por clase')
ax1.set_ylabel('Tasa')
ax1.set_xticklabels(['1ª', '2ª', '3ª'], rotation=0)

# Por sexo
ax2 = fig.add_subplot(gs[0, 2])
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=ax2, color=['#9b59b6', '#e67e22'])
ax2.set_title('Supervivencia por sexo')
ax2.set_ylabel('Tasa')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

# Puerto de embarque
ax3 = fig.add_subplot(gs[0, 3])
df.groupby('embarked')['survived'].mean().plot(kind='bar', ax=ax3, color='teal')
ax3.set_title('Supervivencia por embarque')
ax3.set_ylabel('Tasa')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)

# Distribución de edad
ax4 = fig.add_subplot(gs[1, 0:2])
df[df['survived']==0]['age'].dropna().hist(bins=30, alpha=0.6, color='#e74c3c', label='No sobrevivió', ax=ax4)
df[df['survived']==1]['age'].dropna().hist(bins=30, alpha=0.6, color='#2ecc71', label='Sobrevivió', ax=ax4)
ax4.set_title('Distribución de edad por supervivencia')
ax4.set_xlabel('Edad'); ax4.legend()

# Distribución de tarifa
ax5 = fig.add_subplot(gs[1, 2:4])
df[df['survived']==0]['fare'].hist(bins=40, alpha=0.6, color='#e74c3c', label='No sobrevivió', ax=ax5)
df[df['survived']==1]['fare'].hist(bins=40, alpha=0.6, color='#2ecc71', label='Sobrevivió', ax=ax5)
ax5.set_title('Distribución de tarifa por supervivencia')
ax5.set_xlabel('Tarifa (£)'); ax5.legend()
ax5.set_xlim(0, 300)

plt.suptitle('Análisis Exploratorio Titanic — Visión General', fontsize=13)
plt.tight_layout(); plt.show()

---
## Fase 3 — Limpieza de datos

In [ ]:
print("=== Plan de limpieza ===")
print("age    → 177 missing (20%) → imputar con mediana por clase+sexo")
print("embarked → 2 missing → imputar con moda")
print("deck   → 688 missing (77%) → demasiados, crear categoría 'Desconocida'")
print("fare   → 0 outliers extremos pero revisar")
print("Columnas redundantes: class, alive, embark_town → eliminar (duplican info)")

# Imputar edad con mediana por clase y sexo
mediana_edad = df.groupby(['pclass', 'sex'])['age'].median()
print("\nMediana de edad por clase y sexo:")
print(mediana_edad)

df['age'] = df.groupby(['pclass', 'sex'])['age'].transform(
    lambda x: x.fillna(x.median())
)

# Imputar embarked con moda
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Cubierta: desconocida para missing
df['deck'] = df['deck'].astype(str).replace('nan', 'Desconocida')

# Eliminar columnas redundantes
df = df.drop(columns=['class', 'alive', 'embark_town', 'who', 'adult_male'])

# Verificar
print("\n=== Missing values tras limpieza ===")
print(df.isnull().sum())
print(f"\nDataset limpio: {df.shape}")

# Outliers en fare
Q1, Q3 = df['fare'].quantile([0.25, 0.75])
IQR    = Q3 - Q1
outliers_fare = df[df['fare'] > Q3 + 3*IQR]
print(f"\nOutliers extremos en tarifa (>3×IQR): {len(outliers_fare)}")
print(f"Tarifa máxima: {df['fare'].max():.2f}£  (percentil 99: {df['fare'].quantile(0.99):.2f}£)")
print("→ Mantenemos outliers: representan camarotes de lujo, tienen información")

---
## Fase 4 — Análisis profundo

In [ ]:
print("=== ANÁLISIS 1: Tasa de supervivencia por variables clave ===")

# Crear grupos de edad
df['grupo_edad'] = pd.cut(df['age'], bins=[0, 12, 18, 35, 60, 100],
                          labels=['Niño', 'Adolescente', 'Adulto joven', 'Adulto', 'Mayor'])

# Crear grupos de tarifa
df['grupo_tarifa'] = pd.qcut(df['fare'], q=4,
                             labels=['Económico', 'Normal', 'Superior', 'Lujo'])

# Tabla de supervivencia por variables
vars_analizar = ['pclass', 'sex', 'embarked', 'alone', 'grupo_edad']
for var in vars_analizar:
    tabla = df.groupby(var)['survived'].agg(['mean','count'])
    tabla.columns = ['tasa_supervivencia', 'n']
    tabla['tasa_supervivencia'] = (tabla['tasa_supervivencia'] * 100).round(1)
    print(f"\nSupervivencia por {var}:")
    print(tabla.sort_values('tasa_supervivencia', ascending=False).to_string())

In [ ]:
print("=== ANÁLISIS 2: Interacciones entre variables ===")

# Sexo × Clase
tabla_cruzada = df.groupby(['sex', 'pclass'])['survived'].mean().unstack() * 100
tabla_cruzada = tabla_cruzada.round(1)
tabla_cruzada.columns.name = 'Clase'
print("\nTasa de supervivencia (%) por sexo y clase:")
print(tabla_cruzada)

# Edad × Clase (estadísticas)
print("\nEdad media por clase:")
print(df.groupby('pclass')['age'].agg(['mean','median','std']).round(2))

# Familia a bordo
df['familia'] = df['sibsp'] + df['parch']
df['tam_familia'] = pd.cut(df['familia'], bins=[-1, 0, 3, 10],
                           labels=['Solo', 'Familia pequeña', 'Familia grande'])
print("\nSupervivencia por tamaño de familia:")
print((df.groupby('tam_familia')['survived'].mean() * 100).round(1))

# Tests estadísticos
print("\n=== Tests estadísticos ===")

# Chi-cuadrado: sexo vs supervivencia
tabla_chi = pd.crosstab(df['sex'], df['survived'])
chi2, p_chi, _, _ = stats.chi2_contingency(tabla_chi)
print(f"Chi² sexo vs supervivencia: χ²={chi2:.2f}, p={p_chi:.4f} → {'Significativo' if p_chi<0.05 else 'No significativo'}")

# t-test: edad de supervivientes vs no supervivientes
edad_vivos   = df[df['survived']==1]['age']
edad_muertos = df[df['survived']==0]['age']
t, p_t = stats.ttest_ind(edad_vivos, edad_muertos)
print(f"t-test edad: vivos={edad_vivos.mean():.2f}a vs muertos={edad_muertos.mean():.2f}a, p={p_t:.4f}")

# t-test: tarifa
fare_vivos   = df[df['survived']==1]['fare']
fare_muertos = df[df['survived']==0]['fare']
t2, p_t2 = stats.ttest_ind(fare_vivos, fare_muertos)
print(f"t-test tarifa: vivos={fare_vivos.mean():.2f}£ vs muertos={fare_muertos.mean():.2f}£, p={p_t2:.4f}")

In [ ]:
# Panel de análisis profundo
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Heatmap sexo × clase
tabla_heat = df.groupby(['sex', 'pclass'])['survived'].mean().unstack()
im = axes[0,0].imshow(tabla_heat.values, cmap='RdYlGn', vmin=0, vmax=1)
axes[0,0].set_xticks([0,1,2]); axes[0,0].set_yticks([0,1])
axes[0,0].set_xticklabels(['1ª clase', '2ª clase', '3ª clase'])
axes[0,0].set_yticklabels(tabla_heat.index)
for i in range(2):
    for j in range(3):
        axes[0,0].text(j, i, f"{tabla_heat.values[i,j]:.0%}", ha='center', va='center', fontsize=12)
plt.colorbar(im, ax=axes[0,0])
axes[0,0].set_title('Supervivencia: Sexo × Clase')

# Boxplot edad por clase y supervivencia
datos_box = []
labels_box = []
for clase in [1, 2, 3]:
    datos_box.append(df[(df['pclass']==clase)&(df['survived']==0)]['age'].dropna())
    datos_box.append(df[(df['pclass']==clase)&(df['survived']==1)]['age'].dropna())
    labels_box += [f'{clase}ª\nMuerto', f'{clase}ª\nVivo']
bplot = axes[0,1].boxplot(datos_box, labels=labels_box, patch_artist=True)
colores = ['#e74c3c','#2ecc71'] * 3
for patch, color in zip(bplot['boxes'], colores):
    patch.set_facecolor(color)
axes[0,1].set_title('Edad por clase y supervivencia')

# Tarifa por supervivencia
axes[0,2].hist(df[df['survived']==0]['fare'].clip(0, 200), bins=30, alpha=0.6,
               color='#e74c3c', label='No sobrevivió', density=True)
axes[0,2].hist(df[df['survived']==1]['fare'].clip(0, 200), bins=30, alpha=0.6,
               color='#2ecc71', label='Sobrevivió', density=True)
axes[0,2].set_title('Distribución tarifa (clip 200£)')
axes[0,2].set_xlabel('Tarifa (£)'); axes[0,2].legend()

# Supervivencia por grupo de edad
sup_edad = df.groupby('grupo_edad')['survived'].mean() * 100
sup_edad.plot(kind='bar', ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Supervivencia por grupo de edad')
axes[1,0].set_ylabel('Tasa (%)'); axes[1,0].set_xticklabels(sup_edad.index, rotation=30, ha='right')

# Supervivencia por tamaño de familia
sup_fam = df.groupby('tam_familia')['survived'].mean() * 100
sup_fam.plot(kind='bar', ax=axes[1,1], color='coral')
axes[1,1].set_title('Supervivencia por tamaño de familia')
axes[1,1].set_ylabel('Tasa (%)'); axes[1,1].set_xticklabels(sup_fam.index, rotation=0)

# Correlación numérica
cols_num = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'familia']
corr = df[cols_num].corr()
im2 = axes[1,2].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1,2].set_xticks(range(len(cols_num)))
axes[1,2].set_yticks(range(len(cols_num)))
axes[1,2].set_xticklabels(cols_num, rotation=45, ha='right', fontsize=8)
axes[1,2].set_yticklabels(cols_num, fontsize=8)
for i in range(len(cols_num)):
    for j in range(len(cols_num)):
        axes[1,2].text(j, i, f"{corr.values[i,j]:.2f}", ha='center', va='center', fontsize=7)
plt.colorbar(im2, ax=axes[1,2])
axes[1,2].set_title('Correlaciones numéricas')

plt.suptitle('Análisis Titanic — Patrones de Supervivencia', fontsize=13)
plt.tight_layout(); plt.show()

---
## Fase 5 — Conclusiones

In [ ]:
print("=" * 60)
print(" CONCLUSIONES DEL EDA TITANIC")
print("=" * 60)

print("""
PREGUNTA: ¿Qué factores determinan la supervivencia?

1. SEXO es el factor más determinante:
   - Mujeres: 74% de supervivencia
   - Hombres: 19% de supervivencia
   → Protocolo 'mujeres y niños primero' claramente visible
   → χ² altamente significativo (p < 0.0001)

2. CLASE del billete:
   - 1ª clase: 63% supervivencia
   - 2ª clase: 47% supervivencia
   - 3ª clase: 24% supervivencia
   → Los camarotes de 3ª estaban en cubierta inferior (más lejos de botes)

3. EDAD importa, pero menos de lo esperado:
   - Niños (<12 años): 59% → prioridad en evacuación
   - Adultos jóvenes: menor tasa (muchos hombres)
   - Diferencia de media estadísticamente significativa (p=...)

4. TARIFA (proxy de clase social):
   - Alta correlación con supervivencia
   - Supervivientes pagaron media ~{:.0f}£ vs {:.0f}£ los fallecidos

5. FAMILIA a bordo:
   - Familias pequeñas (1-3): mejor tasa
   - Solos o familias grandes: peor tasa
   → Familias grandes tardaban más en reunirse para evacuar

HIPÓTESIS PARA MODELADO:
  Variables más predictivas: sex, pclass, age, fare, familia
  Posibles interacciones: sex×pclass, age×pclass
  Dato ausente: deck (77% missing, puede ser informativo si se imputa)
""".format(
    df[df['survived']==1]['fare'].mean(),
    df[df['survived']==0]['fare'].mean()
))

# Tabla resumen final
print("RESUMEN NUMÉRICO FINAL:")
resumen_final = {
    'Total pasajeros':     len(df),
    'Supervivientes':      df['survived'].sum(),
    'Tasa supervivencia':  f"{df['survived'].mean()*100:.1f}%",
    'Edad media total':    f"{df['age'].mean():.1f} años",
    'Tarifa media':        f"{df['fare'].mean():.2f}£",
    'Viajaban solos':      f"{df['alone'].sum()} ({df['alone'].mean()*100:.0f}%)",
    'Variables finales':   len(df.columns),
}
for k, v in resumen_final.items():
    print(f"  {k:25}: {v}")

---
## Bonus — Preparación para ML

Convertimos el dataset limpio en un formato listo para entrenar un modelo.

In [ ]:
# Seleccionar y transformar features
df_ml = df[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked', 'alone', 'familia']].copy()

# Codificación de variables categóricas
df_ml['sex_num']      = (df_ml['sex'] == 'female').astype(int)
df_ml['embarked_C']   = (df_ml['embarked'] == 'C').astype(int)
df_ml['embarked_Q']   = (df_ml['embarked'] == 'Q').astype(int)
df_ml['alone_num']    = df_ml['alone'].astype(int)
df_ml = df_ml.drop(columns=['sex', 'embarked', 'alone'])

# Normalización Min-Max de variables continuas
for col in ['age', 'fare', 'familia']:
    mn, mx = df_ml[col].min(), df_ml[col].max()
    df_ml[col + '_norm'] = ((df_ml[col] - mn) / (mx - mn)).round(4)

# Features finales
features = ['pclass', 'sex_num', 'age_norm', 'fare_norm',
            'embarked_C', 'embarked_Q', 'alone_num', 'familia_norm']
X = df_ml[features]
y = df_ml['survived']

print("Dataset listo para ML:")
print(f"X shape: {X.shape}")
print(f"y distribución: {y.value_counts().to_dict()}")
print("\nPrimeras filas de X:")
print(X.head())
print("\nCorrelación features vs target:")
corr_target = X.corrwith(y).sort_values(key=abs, ascending=False)
print(corr_target.round(4))

print("\nEl feature más correlado con sobrevivir: sex_num (mujer)")
print("El feature más anticorrelado: pclass (clase más alta = número menor = mayor supervivencia)")

---
## Práctica

### Ejercicio 1 — EDA del dataset `tips`

**Enunciado:** Realiza un EDA completo del dataset de propinas de restaurante:

```python
tips = sns.load_dataset('tips')
```

Columnas: `total_bill`, `tip`, `sex`, `smoker`, `day`, `time`, `size`

1. Exploración inicial (shape, tipos, missings, describe)
2. ¿Qué % de propina deja cada grupo? (sex, smoker, day, time)
3. Crea columna `pct_propina = tip / total_bill * 100`
4. ¿Hay diferencia significativa en propina entre fumadores y no fumadores?
5. Visualiza: distribución de propina, propina vs total_bill, propina por día

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

tips = sns.load_dataset('tips')

# 1. Exploración
print(f"Shape: {tips.shape}")
print(tips.dtypes)
print(tips.isnull().sum())
print(tips.describe().round(2))

# 2. Propina por grupo
tips['pct_propina'] = (tips['tip'] / tips['total_bill'] * 100).round(2)
for var in ['sex', 'smoker', 'day', 'time']:
    print(f"\nPropina % por {var}:")
    print(tips.groupby(var)['pct_propina'].mean().round(2))

# 4. Test estadístico
fumadores     = tips[tips['smoker']=='Yes']['pct_propina']
no_fumadores  = tips[tips['smoker']=='No']['pct_propina']
t, p = stats.ttest_ind(fumadores, no_fumadores)
print(f"\nt-test fumadores vs no fumadores:")
print(f"  fumadores: {fumadores.mean():.2f}%  no_fumadores: {no_fumadores.mean():.2f}%")
print(f"  t={t:.4f}, p={p:.4f} → {'Significativo' if p<0.05 else 'No significativo'}")

# 5. Visualización
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

tips['pct_propina'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Distribución % propina')
axes[0].set_xlabel('% propina')

axes[1].scatter(tips['total_bill'], tips['tip'], alpha=0.5, color='coral')
m, b = np.polyfit(tips['total_bill'], tips['tip'], 1)
x = np.linspace(tips['total_bill'].min(), tips['total_bill'].max(), 100)
axes[1].plot(x, m*x+b, 'r-', lw=2)
r, _ = stats.pearsonr(tips['total_bill'], tips['tip'])
axes[1].set_title(f'Propina vs Total (r={r:.3f})')
axes[1].set_xlabel('Cuenta total ($)'); axes[1].set_ylabel('Propina ($)')

tips.boxplot(column='pct_propina', by='day', ax=axes[2],
             flierprops=dict(marker='o', ms=3, alpha=0.5))
axes[2].set_title('% Propina por día')
plt.suptitle('')
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 2 — Análisis de correlaciones

**Enunciado:** Sobre el dataset Titanic limpio (`df`) del proyecto:

1. Calcula la correlación de todas las variables numéricas con `survived`
2. Crea una nueva feature: `fare_per_person = fare / (familia + 1)`
3. ¿Mejora la correlación de `fare_per_person` vs `fare` con supervivencia?
4. Visualiza la distribución de `fare_per_person` por clase y supervivencia
5. Interpreta: ¿tiene sentido económico esta nueva variable?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import matplotlib.pyplot as plt
from scipy import stats

# 1. Correlaciones con survived
cols_num = df.select_dtypes(include='number').columns
corr_surv = df[cols_num].corrwith(df['survived']).sort_values(key=abs, ascending=False)
print("Correlación con survived:")
print(corr_surv.round(4))

# 2. Nueva feature
df['fare_per_person'] = df['fare'] / (df['familia'] + 1)

# 3. Comparar correlación
r_fare, _  = stats.pearsonr(df['fare'], df['survived'])
r_fpp, _   = stats.pearsonr(df['fare_per_person'], df['survived'])
print(f"\nCorrelación fare vs survived:           {r_fare:.4f}")
print(f"Correlación fare_per_person vs survived: {r_fpp:.4f}")
print(f"¿Mejora? {'Sí' if abs(r_fpp) > abs(r_fare) else 'No'}")

# 4. Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, var, titulo in zip(axes,
    ['fare', 'fare_per_person'],
    ['Tarifa total', 'Tarifa por persona']):

    for clase in [1, 2, 3]:
        sub = df[df['pclass'] == clase]
        ax.scatter(sub[var].clip(0, 200), sub['survived'] + (clase-2)*0.1,
                   alpha=0.3, label=f'{clase}ª clase', s=15)
    ax.set_title(titulo)
    ax.set_xlabel('£'); ax.set_ylabel('Sobrevivió')
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

# 5. Interpretación
print("\nInterpretación:")
print("  fare_per_person normaliza el precio por el tamaño de familia.")
print("  Un billete de £100 para 5 personas (£20/pers) ≠ £100 para 1 (£100/pers).")
print("  Esta variable puede capturar mejor la riqueza individual que la tarifa total.")
```
</details>

---
### Ejercicio 3 — Perfilado de grupos

**Enunciado:** Identifica los 4 grupos con mayor y menor supervivencia en el Titanic y crea un informe:

1. Agrupa por `sex`, `pclass`, `alone` y calcula tasa de supervivencia + n
2. Filtra grupos con al menos 10 personas
3. Muestra el Top 4 (mayor supervivencia) y el Bottom 4 (menor supervivencia)
4. Para cada extremo, muestra también: edad media, tarifa media
5. Visualiza con un gráfico de barras horizontales, coloreando por tasa

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# 1 y 2. Agrupación y filtro
perfil = df.groupby(['sex', 'pclass', 'alone']).agg(
    tasa=('survived', 'mean'),
    n=('survived', 'count'),
    edad_media=('age', 'mean'),
    tarifa_media=('fare', 'mean')
).round(3).reset_index()

perfil = perfil[perfil['n'] >= 10].sort_values('tasa', ascending=False)

# Etiqueta legible
perfil['grupo'] = perfil.apply(
    lambda r: f"{r['sex']}, {r['pclass']}ª clase, {'solo' if r['alone'] else 'con familia'}",
    axis=1
)

# 3 y 4. Top y Bottom
top4    = perfil.head(4)
bottom4 = perfil.tail(4)

for titulo, df_g in [('TOP 4 (mayor supervivencia)', top4),
                      ('BOTTOM 4 (menor supervivencia)', bottom4)]:
    print(f"\n{titulo}")
    for _, row in df_g.iterrows():
        print(f"  {row['grupo']:45} tasa={row['tasa']:.0%}  n={row['n']:3.0f}  "
              f"edad_media={row['edad_media']:.1f}a  tarifa={row['tarifa_media']:.2f}£")

# 5. Visualización
fig, ax = plt.subplots(figsize=(12, 5))
todos   = pd.concat([top4, bottom4]).sort_values('tasa')
colores = ['#e74c3c' if t < 0.5 else '#2ecc71' for t in todos['tasa']]

bars = ax.barh(todos['grupo'], todos['tasa'] * 100, color=colores)
ax.axvline(50, color='gray', ls='--', lw=1)
for bar, (_, row) in zip(bars, todos.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"n={row['n']:.0f}", va='center', fontsize=9)

ax.set_xlabel('Tasa de supervivencia (%)')
ax.set_title('Perfiles extremos de supervivencia en el Titanic')
ax.set_xlim(0, 115)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 4 — EDA de dataset de IA

**Enunciado:** Genera y analiza un dataset de un sistema de recomendación:

```python
np.random.seed(42)
n_usuarios = 500
eda_df = pd.DataFrame({
    'usuario_id':   range(1, n_usuarios+1),
    'edad':         np.random.normal(35, 12, n_usuarios).clip(18, 70).round(),
    'sesiones':     np.random.poisson(20, n_usuarios),
    'clics':        np.random.poisson(150, n_usuarios),
    'tiempo_min':   np.random.exponential(45, n_usuarios).round(1),
    'compras':      np.random.poisson(3, n_usuarios),
    'segmento':     np.random.choice(['casual','regular','premium'], n_usuarios, p=[0.5,0.35,0.15]),
    'dispositivo':  np.random.choice(['mobile','desktop','tablet'], n_usuarios, p=[0.6,0.3,0.1]),
})
eda_df['tasa_conversion'] = (eda_df['compras'] / eda_df['sesiones']).fillna(0).round(4)
```

Realiza un EDA completo y responde: ¿qué segmento es más valioso para el negocio?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
n = 500
eda_df = pd.DataFrame({
    'usuario_id':   range(1, n+1),
    'edad':         np.random.normal(35, 12, n).clip(18, 70).round(),
    'sesiones':     np.random.poisson(20, n),
    'clics':        np.random.poisson(150, n),
    'tiempo_min':   np.random.exponential(45, n).round(1),
    'compras':      np.random.poisson(3, n),
    'segmento':     np.random.choice(['casual','regular','premium'], n, p=[0.5,0.35,0.15]),
    'dispositivo':  np.random.choice(['mobile','desktop','tablet'], n, p=[0.6,0.3,0.1]),
})
eda_df['tasa_conversion'] = (eda_df['compras'] / eda_df['sesiones']).fillna(0).round(4)

# Exploración
print(f"Shape: {eda_df.shape}")
print(eda_df.describe().round(2))

# Análisis por segmento
valor_seg = eda_df.groupby('segmento').agg(
    n=('usuario_id','count'),
    sesiones_media=('sesiones','mean'),
    compras_media=('compras','mean'),
    conversion=('tasa_conversion','mean'),
    tiempo_media=('tiempo_min','mean')
).round(3)
print("\nMétricas por segmento:")
print(valor_seg)

# Correlaciones
cols_num = ['sesiones','clics','tiempo_min','compras','tasa_conversion']
print("\nCorrelaciones con compras:")
print(eda_df[cols_num].corrwith(eda_df['compras']).round(4).sort_values(key=abs, ascending=False))

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

segmentos = ['casual','regular','premium']
eda_df.boxplot(column='compras', by='segmento', ax=axes[0],
               flierprops=dict(marker='o', ms=3))
axes[0].set_title('Compras por segmento'); plt.sca(axes[0]); plt.title('Compras por segmento')

axes[1].scatter(eda_df['sesiones'], eda_df['compras'], alpha=0.3,
                c=eda_df['segmento'].map({'casual':0,'regular':1,'premium':2}),
                cmap='viridis')
axes[1].set_title('Sesiones vs Compras'); axes[1].set_xlabel('Sesiones'); axes[1].set_ylabel('Compras')

conv_seg = eda_df.groupby('segmento')['tasa_conversion'].mean().reindex(segmentos)
axes[2].bar(segmentos, conv_seg.values * 100, color=['#95a5a6','#3498db','#2ecc71'])
axes[2].set_title('Tasa de conversión por segmento')
axes[2].set_ylabel('%')

plt.tight_layout(); plt.show()

print("\nConclusión: el segmento 'premium' es más valioso:")
print(f"  Mayor tasa de conversión: {eda_df[eda_df['segmento']=='premium']['tasa_conversion'].mean():.3f}")
print(f"  Mayor media de compras:   {eda_df[eda_df['segmento']=='premium']['compras'].mean():.2f}")
print("  Aunque es el 15% de usuarios, genera el mayor valor por persona.")
```
</details>

---
### Ejercicio 5 (reto) — EDA automatizado

**Enunciado:** Implementa una función `eda_automatico(df, target)` que:

1. Genera un informe textual: shape, missings, tipos, estadísticas
2. Para variables numéricas: histograma + correlación con target
3. Para variables categóricas: frecuencias + tasa media del target por categoría
4. Detecta y muestra outliers en variables numéricas
5. Devuelve un dict con las 3 variables más correladas con el target

Pruébala con el dataset Titanic (`df`, target=`survived`) y el de tips (target=`pct_propina`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def eda_automatico(df, target, max_categorias=10):
    print("=" * 60)
    print(f" EDA AUTOMÁTICO  |  Target: '{target}'")
    print("=" * 60)

    # 1. Informe textual
    print(f"\nShape: {df.shape}")
    print(f"Duplicados: {df.duplicated().sum()}")
    print("\nMissing values:")
    miss = df.isnull().sum()
    print(miss[miss > 0].to_string() if miss.sum() > 0 else "  Ninguno")

    cols_num  = df.select_dtypes(include='number').columns.tolist()
    cols_cat  = df.select_dtypes(include=['object','category','bool']).columns.tolist()
    if target in cols_num:  cols_num.remove(target)
    if target in cols_cat:  cols_cat.remove(target)

    print(f"\nVariables numéricas ({len(cols_num)}): {cols_num}")
    print(f"Variables categóricas ({len(cols_cat)}): {cols_cat}")

    # 2. Numéricas
    if cols_num:
        n_plots = len(cols_num)
        fig, axes = plt.subplots(1, n_plots, figsize=(min(n_plots*3, 18), 3))
        if n_plots == 1: axes = [axes]
        corr_num = {}
        for ax, col in zip(axes, cols_num):
            datos = df[col].dropna()
            ax.hist(datos, bins=25, density=True, alpha=0.7, color='steelblue')
            r, p = stats.pearsonr(df[col].dropna(), df.loc[df[col].notna(), target])
            corr_num[col] = round(r, 4)
            ax.set_title(f"{col}\nr={r:.3f}", fontsize=9)
        plt.suptitle("Distribución variables numéricas", fontsize=10)
        plt.tight_layout(); plt.show()

    # 3. Categóricas
    if cols_cat:
        n_plots = len(cols_cat)
        fig, axes = plt.subplots(1, n_plots, figsize=(min(n_plots*4, 18), 3))
        if n_plots == 1: axes = [axes]
        for ax, col in zip(axes, cols_cat):
            tasa = df.groupby(col)[target].mean().sort_values(ascending=False).head(max_categorias)
            tasa.plot(kind='bar', ax=ax, color='coral')
            ax.set_title(f"{col} vs {target}", fontsize=9)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
        plt.suptitle("Media del target por categoría", fontsize=10)
        plt.tight_layout(); plt.show()

    # 4. Outliers
    print("\nOutliers (método IQR):")
    for col in cols_num:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        n_out = ((df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)).sum()
        print(f"  {col:20}: {n_out} outliers ({n_out/len(df)*100:.1f}%)")

    # 5. Top correlaciones
    if cols_num:
        top3 = sorted(corr_num.items(), key=lambda x: abs(x[1]), reverse=True)[:3]
        print("\nTop 3 variables numéricas más correladas con target:")
        for col, r in top3:
            print(f"  {col}: r={r}")
        return dict(top3)
    return {}

# Prueba con Titanic
import seaborn as sns
df_t = sns.load_dataset('titanic').copy()
df_t['age'] = df_t.groupby(['pclass','sex'])['age'].transform(lambda x: x.fillna(x.median()))
df_t['embarked'].fillna('S', inplace=True)
resultado = eda_automatico(df_t[['survived','pclass','age','fare','sex','embarked','alone']], 'survived')
```
</details>

---
## Resumen del Módulo 6

```
┌─────────────────────────────────────────────────────────┐
│              MÓDULO 6 — MATEMÁTICAS PARA IA             │
├──────────┬──────────────────────────────────────────────┤
│ Clase 1  │ Vectores — NumPy, norma, similitud coseno    │
│ Clase 2  │ Matrices — operaciones, transformaciones, PCA│
│ Clase 3  │ Probabilidad — Bayes, distribuciones, Monte C│
│ Clase 4  │ Estadística descriptiva — medidas, outliers  │
│ Clase 5  │ Inferencia — TCL, intervalos, t-test         │
│ Clase 6  │ pandas — Series, DataFrame, limpieza, merge  │
│ Clase 7  │ EDA completo — Titanic, 5 fases, preparar ML │
└──────────┴──────────────────────────────────────────────┘

Pipeline completo que ahora sabes hacer:

  Datos crudos
      ↓ pandas: cargar, explorar
  Datos entendidos
      ↓ estadística descriptiva: resumir, visualizar
  Datos analizados
      ↓ limpieza: NaN, outliers, tipos
  Datos limpios
      ↓ inferencia: ¿diferencias significativas?
  Hipótesis validadas
      ↓ ingeniería de features: normalizar, codificar
  Dataset listo para ML → Módulo 7
```

## ¿Qué viene en el próximo módulo?

**Módulo 7 — Machine Learning** :
Regresión lineal, clasificación, árboles de decisión, evaluación de modelos y cross-validation. Usaremos directamente los datasets que hemos preparado en este módulo.